# 📦 Notebook 01 — Data Understanding
### Walmart Retail Sales Demand Forecasting

---

## 🏪 Business Context

Every week, Walmart has to decide: *how much stock should each store carry?*

Get it right and everything runs smoothly. Get it wrong in either direction and there's a real cost:

| Problem | Cause | Cost |
|---|---|---|
| **Overstock** | Ordered too much | Warehousing, spoilage, tied-up capital |
| **Stockout** | Ordered too little | Lost sales, unhappy customers, churn |

This project builds a **demand forecasting system** that predicts weekly sales per store — accurately enough to meaningfully reduce both problems.

---

## 🎯 Goal of This Notebook

Before building any model, we need to understand our data deeply:
- What does each row represent?
- What are we trying to predict?
- What information do we have to work with?
- Are there any obvious data quality issues?


## 1. Load Libraries & Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Make plots look clean
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

In [ ]:
df = pd.read_csv('../data/walmart_sales.csv')

print(f'Rows: {len(df):,}')
print(f'Columns: {df.shape[1]}')
df.head()

## 2. What Does Each Column Mean?

| Column | Type | Plain English Description |
|---|---|---|
| `Store` | int | Which Walmart store (1–45). Each store has its own patterns. |
| `Date` | string | The week-ending date for that record |
| `Weekly_Sales` | float | ⭐ **This is what we want to predict** — total revenue that week |
| `Holiday_Flag` | int | 1 = holiday week (Christmas, Thanksgiving, etc.), 0 = normal week |
| `Temperature` | float | Regional temperature in °F that week |
| `Fuel_Price` | float | Local fuel price ($/gallon) — affects how often people drive to stores |
| `CPI` | float | Consumer Price Index — a measure of inflation (how much does $1 buy?) |
| `Unemployment` | float | Local unemployment rate — fewer jobs = less spending |

> **One row = one store's full story for one week.** Everything we know about that store, that week, is packed into these 8 columns.


## 3. Inspect the Data

In [ ]:
# Check data types and whether anything is missing
df.info()

In [ ]:
# Check for missing values — none expected, but always good to confirm
missing = df.isnull().sum()
print('Missing values per column:')
print(missing)

if missing.sum() == 0:
    print('\n✅ No missing values — the dataset is complete.')

In [ ]:
# Statistical summary of all numeric columns
# This tells us the range, average, and spread of each variable
df.describe().round(2)

### 💡 What the summary table tells us

- **Weekly_Sales** ranges from ~$200K to ~$3.8M — stores vary dramatically in size
- **Holiday_Flag** mean of ~0.07 confirms that only ~7% of weeks are holiday weeks (makes sense — there are only a handful of major holidays per year)
- **Temperature** ranges from below freezing to ~100°F, reflecting the geographic diversity across stores
- **Unemployment** spans 3.7% to 14.3% — significant regional variation across the 2010–2012 period


## 4. Time Structure of the Data

In [ ]:
# Convert Date from string to proper datetime
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)

# Check the time range
print(f'First date  : {df["Date"].min().date()}')
print(f'Last date   : {df["Date"].max().date()}')
print(f'Total weeks : {df["Date"].nunique()}')
print(f'Total stores: {df["Store"].nunique()}')
print(f'Total rows  : {len(df):,}  (= {df["Date"].nunique()} weeks × {df["Store"].nunique()} stores)')

### 💡 Why weekly data is a good choice

- **Too granular (daily):** Day-to-day sales are noisy. Monday vs Friday can look very different for reasons that have nothing to do with actual demand.
- **Too coarse (monthly):** We lose the ability to spot week-to-week patterns like holiday spikes.
- **Weekly is the sweet spot** — it smooths out daily noise while still capturing seasonal patterns like holiday weeks and back-to-school periods.


## 5. Understanding the Target Variable: Weekly_Sales

In [ ]:
print('Weekly Sales summary across all stores:')
print(f'  Average : ${df["Weekly_Sales"].mean():>12,.0f}')
print(f'  Median  : ${df["Weekly_Sales"].median():>12,.0f}')
print(f'  Min     : ${df["Weekly_Sales"].min():>12,.0f}')
print(f'  Max     : ${df["Weekly_Sales"].max():>12,.0f}')
print(f'  Std Dev : ${df["Weekly_Sales"].std():>12,.0f}')

## 6. Initial Assumptions

Based on what we know about retail and what we've seen in the data so far:

1. **Seasonality matters** — Sales likely spike around Christmas, Thanksgiving, and back-to-school periods. A good model needs to know what time of year it is.

2. **Holidays drive spikes** — The `Holiday_Flag` column should be a strong predictor. We expect significantly higher sales in holiday weeks.

3. **Stores are different** — A large store in a wealthy suburb sells much more than a small store in a rural area. We should build separate models per store rather than one model for all.

4. **Economic factors have some influence** — High unemployment and rising fuel prices could suppress spending. These are worth investigating.

---

## ✅ Summary

| Item | Detail |
|---|---|
| Dataset size | 6,435 rows × 8 columns |
| Stores covered | 45 |
| Time range | Feb 2010 – Oct 2012 (~2.5 years) |
| Missing values | None |
| Target variable | `Weekly_Sales` |
| Key predictors | `Holiday_Flag`, `Store`, `Date` (seasonality), economic variables |

**Next:** Notebook 02 will visualise sales trends, compare stores, and explore what drives sales up or down.
